In [ ]:
%pip install -q numpy pandas pyarrow torch datasets sentence-transformers accelerate

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
# Clone the repository into this Drive folder once, as described in the README.
ROOT = Path("/content/drive/MyDrive/Conversational-History-Retriever")
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
RESULTS = ROOT / "results"
PAIRS_PATH = INTERIM / "pairs.parquet"
REWRITES_PATH = INTERIM / "rewrites.parquet"
for directory in (ROOT, INTERIM, PROCESSED, MODELS, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)
print(f"ROOT: {ROOT}")

In [ ]:
if not PAIRS_PATH.exists():
    raise FileNotFoundError(
        f"Shared pairs file not found: {PAIRS_PATH}. "
        "Run notebooks/data.ipynb through the shared pairs export first."
    )
pairs = pd.read_parquet(PAIRS_PATH)
required = {"qid", "query", "did", "entry"}
missing = required - set(pairs.columns)
if missing:
    raise ValueError(f"Shared pairs table is missing required columns: {sorted(missing)}")
entries = pairs["entry"].astype("string")
if (entries.isna() | entries.str.strip().eq("")).any():
    raise ValueError("Shared pairs table contains missing or blank entries; rerun the data notebook export")
print(f"Loaded {len(pairs):,} pairs from {PAIRS_PATH}")
display(pairs[["qid", "did", "entry"]].head(10))